# Stage 2 reranker — end-to-end pipeline demo

**Goal**: validate the Stage 1 → Stage 2 pipeline on three realistic user journeys, with Stage 1 routed to the best available model per journey.

The Stage 2 formula:
$$\text{final}(u, r) = s_{diet} \times (\alpha_t \cdot s_{taste} + \alpha_p \cdot s_{pantry} + \alpha_n \cdot s_{nutrition})$$
with $\alpha_t + \alpha_p + \alpha_n = 1$ on the simplex.

**What this notebook is NOT**: the α-sweep experiment finding the optimum. That's the deck's headline plot and needs strong Stage 1 (EASE/BPR) + a labelled persona test set. Here we just demonstrate the architecture *runs* in each mode.

**Three user journeys this notebook covers**

| Journey | Stage 1 model | What input the user provided |
|---|---|---|
| **Baseline** (anonymous) | Popularity | nothing — generic top-100 |
| **Persona** (registered with taste history) | SBERT on `taste_seeds` | 8 recipe IDs the persona enjoys |
| **Walk-in** (live demo audience) | SBERT on pantry-text | 5-10 pantry ingredient names |

All three flow into the **same Stage 2 reranker** — the architectural payoff.

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

def _ensure_project_root():
    cwd = Path(os.getcwd()).resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            os.chdir(candidate)
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not locate the PantryPlate project root.")

PROJECT_ROOT = _ensure_project_root()
print(f"Project root: {PROJECT_ROOT}")

from src.data.loader import load_recipes, load_train_interactions
from src.models.popularity import PopularityRecommender
from src.models.sentence_bert import SentenceBERTRecommender
from src.reranker import Stage2Reranker
from src.eval.useful_recall import is_useful, is_macro_near
from src.reranker import diet_compliant, missing_count, get_staples_for_persona

Project root: /Users/ikhyvicky/Documents/MITB_stuff/CS608Project2


## 1. Load personas + recipes

Three personas live in `data/personas/`. As of this branch, each has populated `taste_seeds` (8 recipe IDs representing what they'd rate highly) — that's what makes SBERT viable as a Stage 1 driver for them.

In [2]:
personas = {}
for p_path in sorted(Path("data/personas").glob("*.json")):
    with open(p_path) as f:
        p = json.load(f)
        personas[p["id"]] = p

for pid, p in personas.items():
    print(f"{pid:<20}  restrictions={p['restrictions'] or '[none]'}  "
          f"kcal={p['macro_targets']['calories']}  "
          f"pantry={len(p['pantry'])}  seeds={len(p['taste_seeds'])}")

recipes = load_recipes().set_index(load_recipes()["id"].astype("int64").rename("recipe_id"))
print(f"\nRecipe catalogue: {len(recipes):,} recipes")

family_friendly       restrictions=[none]  kcal=700  pantry=25  seeds=8
fitness_focused       restrictions=[none]  kcal=500  pantry=25  seeds=8
vegan_busy            restrictions=['vegan']  kcal=550  pantry=25  seeds=8



Recipe catalogue: 231,637 recipes


## 2. Stage 1 routing — which model for which journey?

For this demo we route to whatever Stage 1 model best fits each user's available input. The Stage 2 reranker is identical across all three.

```
JOURNEY              STAGE 1 METHOD                          INPUT NEEDED
─────────────────────────────────────────────────────────────────────────────
Baseline (anonymous)  popularity.recommend(_, k=100)          nothing
Persona (registered)  sbert.recommend_for_seeds(seeds, k=100) 5-10 recipe IDs
Walk-in (live demo)   sbert.recommend_for_text(pantry, k=100) 5-10 ingredient strings
```

We fit both Popularity and SBERT once — fits are fast (popularity ~1s, SBERT cache hit ~25s).

In [3]:
train = load_train_interactions()

pop   = PopularityRecommender().fit(train)
sbert = SentenceBERTRecommender(batch_size=256).fit(train)

print(f"Popularity: {len(pop._ranked_items):,} items ranked")
print(f"SBERT:      recipe matrix {sbert._recipe_matrix.shape}")

Popularity: 159,131 items ranked
SBERT:      recipe matrix (231637, 384)


## 3. Generate the Stage 1 candidate pools (top-100 per journey)

The three pools differ in how they were *generated*. They're handed to Stage 2 the same way.

In [4]:
POOL_SIZE = 100

# Journey A: anonymous baseline — generic popular top-100
baseline_ids = pop.recommend(user_id=-1, k=POOL_SIZE, exclude_seen=False)
baseline_scores = {rid: 1.0 / (rank + 1) for rank, rid in enumerate(baseline_ids)}

# Journey B: per-persona candidates from SBERT taste_seeds
persona_pools = {}
for pid, p in personas.items():
    ids = sbert.recommend_for_seeds(p["taste_seeds"], k=POOL_SIZE)
    scores = {rid: 1.0 / (rank + 1) for rank, rid in enumerate(ids)}
    persona_pools[pid] = (ids, scores)

# Journey C: walk-in audience — pretend the volunteer typed a pantry
walkin_pantry = ["chicken breast", "rice", "broccoli", "garlic", "olive oil"]
walkin_restrictions = []
walkin_macros = {"calories": 600}  # they gave a soft calorie hint, nothing else
walkin_ids = sbert.recommend_for_text(walkin_pantry, k=POOL_SIZE)
walkin_scores = {rid: 1.0 / (rank + 1) for rank, rid in enumerate(walkin_ids)}

print("Pool overlap with the baseline (popularity) pool:")
for pid, (ids, _) in persona_pools.items():
    overlap = len(set(ids) & set(baseline_ids))
    print(f"  baseline ∩ persona_{pid:<18}: {overlap:>3}/{POOL_SIZE}")
overlap = len(set(walkin_ids) & set(baseline_ids))
print(f"  baseline ∩ walkin_pantry         : {overlap:>3}/{POOL_SIZE}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pool overlap with the baseline (popularity) pool:
  baseline ∩ persona_family_friendly   :   0/100
  baseline ∩ persona_fitness_focused   :   0/100
  baseline ∩ persona_vegan_busy        :   0/100
  baseline ∩ walkin_pantry         :   0/100


## 4. Stage 2 reranking with default α = (0.5, 0.3, 0.2)

Run all three journeys through the reranker. For walk-in, drop $\alpha_n$ since they gave only a soft calorie target — show how alphas reflect what the user provided.

In [5]:
reranker_full   = Stage2Reranker(alpha_taste=0.5, alpha_pantry=0.3, alpha_nutrition=0.2)
reranker_nomacro = Stage2Reranker(alpha_taste=0.5, alpha_pantry=0.5, alpha_nutrition=0.0)

def name_lookup(rid):
    return recipes.loc[rid, "name"] if rid in recipes.index else "(unknown)"

# Baseline + persona modes both use full alphas (personas have macro targets)
results_top3 = {}

results_top3["baseline (any persona, anon S1)"] = [
    name_lookup(r) for r in
    reranker_full.rerank(personas["fitness_focused"], baseline_ids, baseline_scores, recipes, k=3)
]

for pid, (ids, scores) in persona_pools.items():
    label = f"persona ({pid}, S1=SBERT-seeds)"
    results_top3[label] = [
        name_lookup(r) for r in
        reranker_full.rerank(personas[pid], ids, scores, recipes, k=3)
    ]

# Walk-in: build an ad-hoc persona from the volunteer's input
walkin_persona = {
    "id": "walkin_demo",
    "pantry": walkin_pantry,
    "macro_targets": walkin_macros,
    "restrictions": walkin_restrictions,
    "exclude_from_staples": [],
}
results_top3["walk-in (S1=SBERT-pantry-text)"] = [
    name_lookup(r) for r in
    reranker_nomacro.rerank(walkin_persona, walkin_ids, walkin_scores, recipes, k=3)
]

pd.DataFrame(results_top3, index=["rank 1", "rank 2", "rank 3"]).T

,rank 1,rank 2,rank 3
"baseline (any persona, anon S1)",to die for crock pot roast,the best ever waffles,my no roll pie crust
"persona (family_friendly, S1=SBERT-seeds)",creamy chicken or turkey with pasta,chicken lasagna burritos,creamy chicken and spaghetti casserole
"persona (fitness_focused, S1=SBERT-seeds)",marinated grilled chicken with watermelon salsa,grilled chicken and pasta salad,easy sweet and sour chicken
"persona (vegan_busy, S1=SBERT-seeds)",spinach with chickpeas,greens and garlic,open faced falafel burgers
walk-in (S1=SBERT-pantry-text),olive garden tuscan garlic chicken,garlicky chicken and broccoli risotto,broccoli with a garlic and lemon dressing


## 5. Constraint-score breakdown — `vegan_busy` persona top-5

Shows $(s_{taste}, s_{pantry}, s_{nutrition}, s_{diet})$ per recipe. `final` is the combined score that drove the ranking. Useful for explaining "why this ranking" in the live demo.

In [6]:
demo_persona = personas["vegan_busy"]
ids, scores = persona_pools["vegan_busy"]
scored = reranker_full.rerank(demo_persona, ids, scores, recipes, k=5, return_scores=True)
scored["name"] = scored["recipe_id"].map(name_lookup)
scored[["recipe_id", "name", "s_taste", "s_pantry", "s_nutrition", "s_diet", "final"]].round(3)

,recipe_id,name,s_taste,s_pantry,s_nutrition,s_diet,final
0,98857,spinach with chickpeas,0.192,0.333,0.450,1,0.286
1,99002,greens and garlic,0.242,0.200,0.005,1,0.182
2,180771,open faced falafel burgers,0.082,0.154,0.421,1,0.171
3,86561,veggie medley,0.027,0.300,0.327,1,0.169
4,134231,fruit and vegetable curry,0.043,0.045,0.563,1,0.148


## 6. α sensitivity — same persona, four α settings

Walks the simplex corners + the default for `fitness_focused`. Different alphas → visibly different top-5s.

In [7]:
demo_pid = "fitness_focused"
ids, scores = persona_pools[demo_pid]
demo_p = personas[demo_pid]

alpha_settings = {
    "taste only (1, 0, 0)":      (1.0, 0.0, 0.0),
    "pantry only (0, 1, 0)":     (0.0, 1.0, 0.0),
    "macros only (0, 0, 1)":     (0.0, 0.0, 1.0),
    "default (0.5, 0.3, 0.2)":   (0.5, 0.3, 0.2),
}

sweep = {}
for label, (at, ap, an) in alpha_settings.items():
    r = Stage2Reranker(alpha_taste=at, alpha_pantry=ap, alpha_nutrition=an)
    top = r.rerank(demo_p, ids, scores, recipes, k=5)
    sweep[label] = [name_lookup(r) for r in top]

pd.DataFrame(sweep, index=[f"rank {i+1}" for i in range(5)]).T

,rank 1,rank 2,rank 3,rank 4,rank 5
"taste only (1, 0, 0)",marinated grilled chicken with watermelon salsa,grilled chicken and pasta salad,chicken in sweet and hot pepper sauce,chicken with balsamic succotash,chicken satay
"pantry only (0, 1, 0)",easy sweet and sour chicken,chicken stuff,almond chicken,grilled chicken with mango habanero glaze,spicy chicken tenders with blue cheese dipping...
"macros only (0, 0, 1)",chicken broccoli and rice casserole,grilled chicken and pasta salad,chicken with black beans and rice,testament chicken rice casserole,chicken strips with sweet and sour sauce
"default (0.5, 0.3, 0.2)",marinated grilled chicken with watermelon salsa,grilled chicken and pasta salad,easy sweet and sour chicken,chicken in sweet and hot pepper sauce,chicken with balsamic succotash


## 7. Useful-Recall coverage — per-constraint pass rates

For each journey, what fraction of the candidate pool satisfies each constraint individually, and all three together? This is the upper bound on Useful Recall@100.

Compares the three Stage 1 strategies side-by-side.

In [8]:
def constraint_coverage(candidate_ids, persona, label):
    diet_pass = pantry_pass = macro_pass = all_pass = 0
    staples = get_staples_for_persona(persona)
    for rid in candidate_ids:
        if rid not in recipes.index: continue
        row = recipes.loc[rid]
        ings = row.get("ingredients_parsed") or []
        tags = row.get("tags_parsed") or []
        nut = row.get("nutrition_parsed") or {}
        d = diet_compliant(ings, tags, persona["restrictions"])
        p = missing_count(ings, persona["pantry"], staples=staples) <= 3
        m = is_macro_near(nut, persona["macro_targets"], tolerance=0.2) if persona["macro_targets"] else True
        if d: diet_pass += 1
        if p: pantry_pass += 1
        if m: macro_pass += 1
        if d and p and m: all_pass += 1
    return {"journey": label, "diet": f"{diet_pass}/100",
            "pantry≤3miss": f"{pantry_pass}/100",
            "macros±20%": f"{macro_pass}/100", "ALL": f"{all_pass}/100"}

rows = []
# Anonymous baseline — measure against each persona for comparison
for pid in personas:
    rows.append(constraint_coverage(baseline_ids, personas[pid],
                                     f"baseline → {pid}"))
# Persona mode
for pid, (ids, _) in persona_pools.items():
    rows.append(constraint_coverage(ids, personas[pid],
                                     f"persona  → {pid}"))
# Walk-in
rows.append(constraint_coverage(walkin_ids, walkin_persona, "walk-in  → walkin_demo"))

pd.DataFrame(rows).set_index("journey")

,diet,pantry≤3miss,macros±20%,ALL
journey,,,,
baseline → family_friendly,100/100,37/100,0/100,0/100
baseline → fitness_focused,100/100,28/100,0/100,0/100
baseline → vegan_busy,1/100,14/100,0/100,0/100
persona → family_friendly,100/100,1/100,0/100,0/100
persona → fitness_focused,100/100,1/100,0/100,0/100
persona → vegan_busy,33/100,0/100,0/100,0/100
walk-in → walkin_demo,100/100,18/100,22/100,0/100


## 8. Notes + limitations

**What this notebook validates** ✓
- Three distinct user journeys flow into the same Stage 2 reranker
- Stage 1 routing works: anonymous → popularity, persona → SBERT-on-seeds, walk-in → SBERT-on-pantry-text
- Persona-mode candidate pools have very low overlap with the baseline (popularity) pool — SBERT is doing real content matching, not regression to popularity
- α sweep across simplex corners produces visibly different top-5s
- Constraint-score breakdown is interpretable (good for the live demo Q&A: "why is this recipe #1?")
- Per-constraint coverage diagnostic shows where each Stage 1 strategy wins and loses

**Honest finding — coverage still depends on Stage 1**

The per-constraint table above tells the architectural story. Anonymous baseline (popularity) produces near-zero coverage for *vegan_busy* (very few popular recipes are vegan) and zero macro hits across all personas. Persona-mode (SBERT-seeded) has higher coverage on the constraints that matter to each persona, because the candidate pool was already *content-similar* to recipes they enjoy.

This is the architectural payoff for routing Stage 1: **smarter upstream → cheaper downstream**. Stage 2 can do more work with a better pool.

**What this notebook deliberately does NOT validate** (deferred)
- **The α-sweep finding the optimum** — needs strong CF Stage 1 (EASE/BPR) and a labelled persona test set. Deck headline experiment.
- **End-to-end Useful Recall@K aggregated across users** — needs per-user evaluation; pending W5.
- **Streamlit demo widget** — slide 17. Build on top of this same architecture; three α sliders + persona switcher + pantry input. ~half day later.

**Known gaps before final demo**
1. ✓ taste_seeds for the 3 personas
2. ✓ Stage 1 routing demonstrated for 3 journeys
3. ⬜ Build EASE / BPR (`docs/week2_onboarding.md` §4b) for the warm-track CF spine
4. ⬜ Useful-Recall harness across personas + α-sweep
5. ⬜ Streamlit widget

**Reference docs**
- Constraint scorers: `src/reranker/scores.py`
- Reranker formula: `src/reranker/combiner.py`
- Useful Recall: `src/eval/useful_recall.py`
- SBERT seed-based methods: `src/models/sentence_bert.py` (`recommend_for_seeds`, `recommend_for_text`)
- Locked decisions: `docs/data_decisions.md` §§7-9